# Deviation Analytics — Summary Statistics & Visual Report

**Position in the pipeline:** runs *after* `topic_modeling` and `root_cause`. This is a **read-only reporting layer** — it consumes the Delta tables already written by the upstream NLP + semantic-search notebooks and renders summary statistics and figures. It does **not** re-run any model or reprocess the raw deviation data.

```
build_reference_glossary  →  deviation_embeddings + AI Search indexes
                          →  ( topic_modeling , root_cause , unblinding_semantic_search )
                          →  THIS REPORT
```

**Tables consumed**
- `deviation_supplementary` — per-event resolved entities (clinical IDs, documents/SOPs, acronyms, CROs, devices, vendors) + labels
- `deviation_topics` / `deviation_topic_assignments` — BERTopic zero-shot topics (int id + readable labels)
- `deviation_root_cause` — LLM Ishikawa (6M) root causes + confidence
- `unblinding_cohort` — pre-built hybrid-retrieval semantic cohort

> **Note:** the source has no *process-step* field, so "which SOP at which step" is not derivable. The report shows the specific most-deviated SOP documents instead.

In [0]:
# Cell 2 — Imports, config, theme, helpers
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ---- Config: consume ONLY tables already written by the upstream notebooks ----
CATALOG, ALYT = "us_gmsgq_dev", "gms_us_alyt"
MART         = "gms_us_mart"
SOURCE_TABLE = f"{CATALOG}.{MART}.tw_deviation_data_formatted_rdq"   # raw deviations (date/row grain)
SUPP_TABLE   = f"{CATALOG}.{MART}.deviation_supplementary"       # entity arrays + labels
TOPICS_TABLE = f"{CATALOG}.{MART}.deviation_topics"              # pr_id, topic(int), topic_prob
TOPIC_ASSIGN = f"{CATALOG}.{MART}.deviation_topic_assignments"   # pr_id, primary/secondary/tertiary label
RC_TABLE     = f"{CATALOG}.{MART}.deviation_root_cause"          # pr_id, root_cause_category, confidence, ...
UNBLINDING   = f"{CATALOG}.{MART}.unblinding_cohort"             # pre-built semantic-search cohort
EMBED_INPUT  = f"{CATALOG}.{MART}.deviation_embed_input"

READ_GROUP = "GMSGQ-Users-General-Read-Access"
def write_and_grant(df, table_name, catalog="us_gmsgq_dev", schema="gms_us_mart", mode="overwrite"):
    fqn = f"{catalog}.{schema}.{table_name}"
    (df.write.mode(mode).option("overwriteSchema", "true").saveAsTable(fqn))
    spark.sql(f"GRANT SELECT ON TABLE {fqn} TO `{READ_GROUP}`")
    print(f"✅ wrote + granted {fqn}")
    display(spark.sql(f"SHOW GRANTS ON TABLE {fqn}"))
    return fqn
    
# AI Search stack (used ONLY by the optional live-search appendix cell)
EMB_INDEX_FINE = f"{CATALOG}.{MART}.deviation_emb_fine_idx"
VS_ENDPOINT    = "deviation-retrieval-vs"
MAX_LEN        = 8192

# ---- Pretty defaults ----
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#cfcfcf", "axes.grid": True, "grid.color": "#ececec",
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titleweight": "bold", "figure.dpi": 110,
})
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3",
           "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]
CAT_COLORS = {
    "Clinical / GCP": "#4C72B0", "Data Integrity & IT": "#64B5CD",
    "Manufacturing / GMP": "#DD8452", "Equipment & Facilities": "#937860",
    "Quality Control": "#CCB974", "Quality Systems": "#55A868",
    "Supply Chain": "#8172B3", "Safety & PV": "#C44E52", "Other / Emergent": "#8C8C8C",
}


def to_list(x):
    """Normalize a Spark array cell (list / np.ndarray / None / scalar) to a clean list."""
    if x is None:
        return []
    if isinstance(x, (list, tuple, np.ndarray)):
        return [e for e in x if e is not None and str(e).strip() != ""]
    try:
        if pd.isna(x):
            return []
    except (TypeError, ValueError):
        pass
    return [x]


def barh_top(ax, series, title, xlabel="Events", color="#4C72B0", n=15, wrap=46):
    """Horizontal bar of the top-n of a value_counts-style Series, with value labels."""
    s = series.head(n)[::-1]
    bars = ax.barh([str(i)[:wrap] for i in s.index], s.values, color=color)
    ax.set_title(title); ax.set_xlabel(xlabel); ax.tick_params(axis="y", labelsize=8)
    for b, v in zip(bars, s.values):
        ax.text(b.get_width(), b.get_y() + b.get_height() / 2, f"  {int(v):,}",
                va="center", ha="left", fontsize=8, color="#333333")
    ax.margins(x=0.14)


def topic_category(label):
    """Map a topic label to a hard-coded high-level category (keyword rules; robust to truncation)."""
    l = str(label or "").lower()
    rules = [
        (("unblind", "protocol", "gcp"),                                            "Clinical / GCP"),
        (("data integrity", "computerized", "it system", "validation"),             "Data Integrity & IT"),
        (("manufactur", "batch", "label", "packaging", "contamination",
          "cleaning", "environmental"),                                             "Manufacturing / GMP"),
        (("equipment", "instrument", "calibration", "malfunction"),                 "Equipment & Facilities"),
        (("stability", "specification", "material", "component"),                   "Quality Control"),
        (("audit", "inspection", "capa", "document control", "sop", "training"),    "Quality Systems"),
        (("vendor", "supplier", "transport", "cold chain"),                         "Supply Chain"),
        (("complaint", "adverse", "pharmacovig", "safety"),                         "Safety & PV"),
    ]
    for keys, cat in rules:
        if any(k in l for k in keys):
            return cat
    return "Other / Emergent"


print("Setup complete. Consuming pre-written tables only (no raw reprocessing).")

In [0]:
# Cell 3 — Load pre-written tables and assemble one per-event master frame
def _load(name):
    return spark.table(name).toPandas()

supp   = _load(SUPP_TABLE).rename(columns={"Event_Number": "pr_id"})
topics = _load(TOPICS_TABLE)
assign = _load(TOPIC_ASSIGN)
rc     = _load(RC_TABLE)
for _df in (supp, topics, assign, rc):
    _df["pr_id"] = _df["pr_id"].astype(str)

ENT_COLS = ["pr_id", "clinical_ids", "documents", "acronyms", "cros", "devices",
            "vendors", "personnel", "LOC", "deterministic_context", "source_free_text_full"]
master = supp[[c for c in ENT_COLS if c in supp.columns]].copy()

master = (master
    .merge(assign[["pr_id", "primary_topic", "secondary_topic", "tertiary_topic"]],
           on="pr_id", how="left")
    .merge(topics[["pr_id", "topic", "topic_prob"]], on="pr_id", how="left")
    .merge(rc[["pr_id", "root_cause_category", "confidence", "contributing_factors"]],
           on="pr_id", how="left"))

# ---- Derived fields ----
master["topic"]          = master["topic"].fillna(-1).astype(int)
master["is_outlier"]     = master["topic"].eq(-1)
master["topic_label"]    = master["primary_topic"].where(
    master["primary_topic"].notna() & (master["topic"] != -1), "Outlier / Unclassified")
master["topic_category"] = master["topic_label"].apply(topic_category)
master["n_clinical"]     = master["clinical_ids"].apply(lambda x: len(to_list(x))) if "clinical_ids" in master else 0
master["n_docs"]         = master["documents"].apply(lambda x: len(to_list(x)))    if "documents"    in master else 0
_ARR_COLS = [c for c in ["clinical_ids", "documents", "acronyms", "cros", "devices", "vendors"] if c in master]
master["n_entities"]     = master[_ARR_COLS].apply(
    lambda row: sum(len(to_list(v)) for v in row), axis=1)

# ---- Exploded entity frequencies (reused downstream) ----
def _explode_freq(col):
    if col not in master:
        return pd.Series(dtype="int64")
    return pd.Series([str(e) for lst in master[col].apply(to_list) for e in lst],
                     dtype=object).value_counts()

clinical_freq = _explode_freq("clinical_ids")
doc_freq      = _explode_freq("documents")
sop_freq      = doc_freq[doc_freq.index.to_series().str.upper().str.startswith("SOP")] \
                if doc_freq.size else doc_freq

N = len(master)
print(f"Master assembled: {N:,} deviations x {master.shape[1]} columns")
print(f"  clinical-ID mentions : {int(clinical_freq.sum()):,}  ({clinical_freq.size:,} distinct)")
print(f"  document mentions    : {int(doc_freq.sum()):,}  ({doc_freq.size:,} distinct; {sop_freq.size:,} distinct SOPs)")
print(f"  non-outlier topics   : {int((~master['is_outlier']).sum()):,}  |  "
      f"root causes identified: {int(master['root_cause_category'].notna().sum()):,}")

## 1 · Summary statistics & entity extraction

How many clinical IDs and SOPs/documents did the deterministic resolver pull out of the corpus, and how are resolved entities distributed across events.

In [0]:
# Cell 5 — Figure A: summary statistics & entity extraction
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Deviation Corpus — Summary Statistics & Entity Extraction", fontsize=15, fontweight="bold")

# (0,0) KPI tiles
ax = axes[0, 0]; ax.axis("off"); ax.set_title("Key figures", loc="left")
kpis = [
    ("Deviations", f"{N:,}"),
    ("Clinical-ID mentions", f"{int(clinical_freq.sum()):,}"),
    ("Distinct clinical IDs", f"{clinical_freq.size:,}"),
    ("Events citing a document", f"{int((master['n_docs'] > 0).sum()):,}"),
    ("Distinct SOPs", f"{sop_freq.size:,}"),
    ("Distinct documents", f"{doc_freq.size:,}"),
]
for i, (lbl, val) in enumerate(kpis):
    r, c = divmod(i, 2)
    x, y = 0.02 + c * 0.50, 0.80 - r * 0.30
    col = PALETTE[i % len(PALETTE)]
    ax.add_patch(Rectangle((x, y - 0.16), 0.46, 0.26, transform=ax.transAxes,
                           facecolor=col, alpha=0.12, edgecolor=col, linewidth=1.2))
    ax.text(x + 0.03, y + 0.02, val, transform=ax.transAxes, fontsize=21, fontweight="bold",
            color=col, va="center")
    ax.text(x + 0.03, y - 0.10, lbl, transform=ax.transAxes, fontsize=9, color="#444", va="center")

# (0,1) most-referenced clinical IDs
if clinical_freq.size:
    barh_top(axes[0, 1], clinical_freq, "Most-referenced clinical IDs", color="#4C72B0", n=12)
else:
    axes[0, 1].axis("off"); axes[0, 1].set_title("No clinical IDs extracted")

# (1,0) most-referenced documents / SOPs
if doc_freq.size:
    barh_top(axes[1, 0], doc_freq, "Most-referenced documents (SOP / spec / form)", color="#DD8452", n=12)
else:
    axes[1, 0].axis("off"); axes[1, 0].set_title("No documents extracted")

# (1,1) resolved-entities-per-event distribution
ax = axes[1, 1]
ax.hist(master["n_entities"].clip(upper=15), bins=range(0, 17),
        color="#55A868", edgecolor="white", align="left")
ax.set_title("Resolved entities per deviation")
ax.set_xlabel("# entities (capped at 15)"); ax.set_ylabel("Events")

plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

## 2 · Topic landscape, outliers & categorization

The common topics and their counts (the "topic map"), the share of events BERTopic left as outliers, and a bar chart of the topics rolled up into a hard-coded business taxonomy (the categorization slide).

In [0]:
# Cell 7 — Figure B: topic counts, outliers & categorization
topic_counts = master.loc[~master["is_outlier"], "topic_label"].value_counts()
cat_counts   = master.loc[~master["is_outlier"], "topic_category"].value_counts()
n_out        = int(master["is_outlier"].sum())
classified   = N - n_out

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.suptitle("Topic Landscape — Counts, Outliers & Categorization", fontsize=15, fontweight="bold")

# left: common topics + counts
barh_top(axes[0], topic_counts, "Common topics (primary assignment)", color="#4C72B0", n=15)

# middle: classified vs outlier donut
axes[1].pie([classified, n_out], labels=["Classified", "Outlier / Unclassified"],
            autopct=lambda p: f"{p:.0f}%\n({int(round(p * N / 100)):,})",
            colors=["#55A868", "#C44E52"], startangle=90,
            wedgeprops=dict(width=0.42, edgecolor="white"), textprops=dict(fontsize=9))
axes[1].set_title(f"Outlier share — {classified:,} classified / {n_out:,} outlier")

# right: categorization slide
cc = cat_counts[::-1]
axes[2].barh(range(len(cc)), cc.values, color=[CAT_COLORS.get(k, "#8C8C8C") for k in cc.index])
axes[2].set_yticks(range(len(cc))); axes[2].set_yticklabels(cc.index)
axes[2].set_title("Topic categorization (hard-coded taxonomy)"); axes[2].set_xlabel("Events")
for i, v in enumerate(cc.values):
    axes[2].text(v, i, f"  {int(v):,}", va="center", fontsize=8, color="#333")
axes[2].margins(x=0.14)

plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

## 3 · Event heatmap — topic × root cause

Where events concentrate across the two independent NLP signals: the BERTopic topic (what the deviation is about) versus the LLM Ishikawa 6M root cause (why it happened).

In [0]:
# Cell 9 — Figure C: event heatmap (topic x root cause)
top_topics = master.loc[~master["is_outlier"], "topic_label"].value_counts().head(12).index
sub = master[master["topic_label"].isin(top_topics) & master["root_cause_category"].notna()]

if len(sub):
    ct = pd.crosstab(sub["topic_label"].str.slice(0, 42), sub["root_cause_category"])
    ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]   # busiest topics on top

    fig, ax = plt.subplots(figsize=(max(9, 1.1 * ct.shape[1] + 4), max(6, 0.55 * ct.shape[0] + 2)))
    im = ax.imshow(ct.values, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(ct.shape[1])); ax.set_xticklabels(ct.columns, rotation=35, ha="right", fontsize=9)
    ax.set_yticks(range(ct.shape[0])); ax.set_yticklabels(ct.index, fontsize=9)
    ax.set_title("Event Heatmap — Topic x Root-Cause Category", fontsize=14, fontweight="bold")
    ax.grid(False)
    vmax = ct.values.max()
    for i in range(ct.shape[0]):
        for j in range(ct.shape[1]):
            v = int(ct.values[i, j])
            if v:
                ax.text(j, i, v, ha="center", va="center", fontsize=8,
                        color="white" if v > vmax * 0.6 else "#333")
    fig.colorbar(im, ax=ax, shrink=0.8, label="Events")
    plt.tight_layout(); plt.show()
else:
    print("Not enough overlapping topic + root-cause data to build the heatmap.")

## 4 · Root causes, recurring signatures & most-deviated SOPs

Common root causes and their counts; the recurring *topic ▸ root-cause* signatures that flag systemic issues; and the specific SOP documents that get deviated from most often.

In [0]:
# Cell 11 — Figure D: root causes, recurring signatures & most-deviated SOPs
rc_counts = master["root_cause_category"].fillna("Not identified").value_counts()

# recurring "signatures" = (topic, root cause) pairs seen most often (systemic recurrence)
sig = (master.loc[~master["is_outlier"] & master["root_cause_category"].notna()]
             .assign(sig=lambda d: d["topic_label"].str.slice(0, 28) + "  \u25b8  " + d["root_cause_category"])
             ["sig"].value_counts())

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle("Root Causes, Recurring Signatures & Most-Deviated SOPs", fontsize=15, fontweight="bold")

barh_top(axes[0], rc_counts, "Root-cause categories (Ishikawa 6M)", color="#C44E52", n=10)
barh_top(axes[1], sig, "Recurring topic \u25b8 root-cause signatures", color="#8172B3", n=12)
if sop_freq.size:
    barh_top(axes[2], sop_freq, "Most-deviated SOPs (specific document)", color="#DD8452", n=12)
else:
    axes[2].axis("off"); axes[2].set_title("No SOP references found")

plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()
print("Note: the source has no process-step field, so 'which step' cannot be reported; "
      "the specific SOP document is shown instead.")

## 5 · Coverage, analyst effort & validation decisions

The automation use-case: how much of the corpus the pipeline covers end-to-end, the analyst effort that auto-triage saves, and the validation funnel that separates high-confidence auto-accepts from events needing human review.

In [0]:
# Cell 13 — Figure E: coverage, analyst effort & validation decisions
n_ent   = int((master["n_entities"] > 0).sum())
n_topic = int((~master["is_outlier"]).sum())
_rc_bad = {"Root Cause - Not Identified", "Other / Indeterminate", "ERROR"}
n_rc    = int((master["root_cause_category"].notna() & ~master["root_cause_category"].isin(_rc_bad)).sum())

conf    = master["confidence"].astype(str).str.lower()
n_high  = int((conf.str.contains("high") | (pd.to_numeric(master["confidence"], errors="coerce") >= 80)).sum())

MIN_PER_EVENT = 20                              # assumed analyst minutes to manually triage + root-cause one event
hours_saved   = n_topic * MIN_PER_EVENT / 60.0

fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
fig.suptitle("Coverage \u00b7 Analyst Effort \u00b7 Validation Decisions", fontsize=15, fontweight="bold")

# coverage funnel
stages = ["All deviations", "\u22651 resolved entity", "Topic assigned", "Root cause identified"]
vals   = [N, n_ent, n_topic, n_rc]
axes[0].barh(range(len(stages))[::-1], vals, color=PALETTE[:4])
axes[0].set_yticks(range(len(stages))[::-1]); axes[0].set_yticklabels(stages)
axes[0].set_title("Automation coverage funnel"); axes[0].set_xlabel("Events")
for i, v in enumerate(vals[::-1]):
    axes[0].text(v, i, f"  {v:,} ({100 * v / N:.0f}%)", va="center", fontsize=8)
axes[0].margins(x=0.20)

# effort tiles
ax = axes[1]; ax.axis("off"); ax.set_title("Estimated analyst effort saved")
tiles = [(f"{n_topic:,}", "events auto-triaged"),
         (f"{MIN_PER_EVENT} min", "assumed manual effort / event"),
         (f"{hours_saved:,.0f} h", "analyst-hours saved"),
         (f"{hours_saved / 8:,.0f} d", "\u2248 analyst-days saved (8 h)")]
for i, (big, small) in enumerate(tiles):
    y = 0.82 - i * 0.22
    ax.text(0.05, y, big, fontsize=20, fontweight="bold", color=PALETTE[i % len(PALETTE)], transform=ax.transAxes)
    ax.text(0.05, y - 0.07, small, fontsize=10, color="#444", transform=ax.transAxes)

# validation decision funnel
dec = ["Model output", "Auto-classified", "High-confidence\n(auto-accept)"]
dv  = [N, n_topic, n_high]
axes[2].bar(range(len(dec)), dv, color=["#8C8C8C", "#4C72B0", "#55A868"])
axes[2].set_xticks(range(len(dec))); axes[2].set_xticklabels(dec, fontsize=9)
axes[2].set_title("Validation decision funnel"); axes[2].set_ylabel("Events")
for i, v in enumerate(dv):
    axes[2].text(i, v, f"{v:,}\n{100 * v / N:.0f}%", ha="center", va="bottom", fontsize=8)
axes[2].margins(y=0.16)
axes[2].text(0.5, -0.24, f"Remaining {N - n_high:,} events routed to analyst review",
             transform=axes[2].transAxes, ha="center", fontsize=8, color="#666")

plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

## 6 · Semantic search — unblinding & protocol deviations

Two high-priority use cases surfaced by the semantic layer. **Unblinding** comes straight from the pre-built `unblinding_cohort` (hybrid vector + keyword retrieval). **Protocol deviations** reuse the BERTopic assignment (the semantic topic signal) — no new pass over raw text.

In [0]:
# Cell 15 — Figure F: semantic-search cohorts (unblinding + protocol deviations)
try:
    unb = spark.table(UNBLINDING).toPandas()
    unb["pr_id"] = unb["pr_id"].astype(str)
except Exception as e:
    unb = pd.DataFrame(); print(f"unblinding_cohort not available: {e}")

# protocol-deviation cohort = any topic slot mentioning "protocol" (semantic topic signal)
prot_mask = (master[["primary_topic", "secondary_topic", "tertiary_topic"]]
             .astype(str)
             .apply(lambda c: c.str.contains("protocol", case=False, na=False))
             .any(axis=1))
prot = master[prot_mask]

fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
fig.suptitle("Semantic-Search Cohorts — Unblinding & Protocol Deviations", fontsize=15, fontweight="bold")

# (0) unblinding provenance
if len(unb) and {"hit_semantic", "hit_keyword"}.issubset(unb.columns):
    so = int((unb["hit_semantic"] & ~unb["hit_keyword"]).sum())
    ko = int((~unb["hit_semantic"] & unb["hit_keyword"]).sum())
    bo = int((unb["hit_semantic"] & unb["hit_keyword"]).sum())
    axes[0].pie([so, ko, bo], labels=["Semantic only", "Keyword only", "Both"],
                autopct="%1.0f%%", colors=["#4C72B0", "#DD8452", "#55A868"], startangle=90,
                wedgeprops=dict(width=0.42, edgecolor="white"), textprops=dict(fontsize=9))
    axes[0].set_title(f"Unblinding cohort provenance\n(n={len(unb):,}, {100 * len(unb) / N:.1f}% of corpus)")
else:
    axes[0].axis("off"); axes[0].set_title("Unblinding cohort unavailable")

# (1) unblinding root causes (prefer the cohort's source column; fall back to the topic slice)
if len(unb) and "Root_Cause_Category" in unb.columns:
    barh_top(axes[1], unb["Root_Cause_Category"].fillna("Unknown").value_counts(),
             "Unblinding — root causes", color="#C44E52", n=8)
else:
    unb_topic = master.loc[master["topic_label"].str.contains("unblind", case=False, na=False),
                           "root_cause_category"].fillna("Unknown").value_counts()
    barh_top(axes[1], unb_topic, "Unblinding topic — root causes", color="#C44E52", n=8)

# (2) protocol-deviation cohort by root cause
barh_top(axes[2], prot["root_cause_category"].fillna("Unknown").value_counts(),
         f"Protocol deviations — root causes (n={len(prot):,})", color="#4C72B0", n=8)

plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.show()
print(f"Unblinding cohort: {len(unb):,} events  |  "
      f"Protocol-deviation cohort (topic signal): {len(prot):,} events")

## 7 · Top offending vendors & program/protocol recurrence

Vendors come from the already-resolved entity arrays. Program- and protocol-level recurrence (which programs/protocols accumulate the most deviations) needs `Program_Number` / `Study_Protocol`, so this panel joins back to `SOURCE_TABLE` and compresses it to one row per `pr_id`.

In [0]:
# Cell 16 — Figure G: top offending vendors & program/protocol recurrence
from pyspark.sql import functions as F

_NA = {"", "N/A", "NA", "N-A", "NONE", "NULL", "-", "--"}
def _clean_counts(series):
    s = series.dropna().astype(str).str.strip()
    return s[~s.str.upper().isin(_NA)].value_counts()

vendor_freq = _explode_freq("vendors")

# program/protocol recurrence — join back to SOURCE_TABLE, compress to pr_id grain
try:
    _pp = (spark.table(SOURCE_TABLE)
                .groupBy(F.col("Event_Number").cast("string").alias("pr_id"))
                .agg(F.first("Program_Number", ignorenulls=True).alias("program_number"),
                     F.first("Study_Protocol",  ignorenulls=True).alias("study_protocol"))
                .toPandas())
    _pp["pr_id"] = _pp["pr_id"].astype(str)
    _pp = _pp[_pp["pr_id"].isin(set(master["pr_id"]))]          # only analyzed events
    prog_freq, proto_freq = _clean_counts(_pp["program_number"]), _clean_counts(_pp["study_protocol"])
    have_pp = True
except Exception as e:
    prog_freq = proto_freq = pd.Series(dtype="int64"); have_pp = False
    print(f"SOURCE_TABLE join unavailable: {e}")

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.suptitle("Top Offending Vendors & Program / Protocol Recurrence", fontsize=15, fontweight="bold")

if vendor_freq.size:
    barh_top(axes[0], vendor_freq, "Top offending vendors", color="#8172B3", n=12)
else:
    axes[0].axis("off"); axes[0].set_title("No vendors extracted")

if have_pp and prog_freq.size:
    barh_top(axes[1], prog_freq, "Programs with most deviations (recurrence)", color="#DD8452", n=12)
else:
    axes[1].axis("off"); axes[1].set_title("Program data unavailable")

if have_pp and proto_freq.size:
    barh_top(axes[2], proto_freq, "Protocols with most deviations (recurrence)", color="#55A868", n=12)
else:
    axes[2].axis("off"); axes[2].set_title("Protocol data unavailable")

plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

if have_pp:
    print(f"Recurrence: {int((prog_freq >= 2).sum()):,} programs and "
          f"{int((proto_freq >= 2).sum()):,} protocols have \u22652 deviations (repeat-offender candidates).")

## 8 · SME validation, severity & review worklists

Built for reviewers: does the model agree with the human root-cause call, which events carry safety/regulatory severity, which events need human validation, and — most importantly — **the actual events behind every chart** so an SME can drill straight into the records. This section joins `SOURCE_TABLE` once to read the human `Root_Cause_Category`, `Impact_Assessment`, `Action_Text`, and program/protocol (read-only, no reprocessing).

In [0]:
# Cell 17 — SME enrichment: join SOURCE_TABLE once + derive severity, agreement & triage
from pyspark.sql import functions as F

# ---- One read-only join back to SOURCE_TABLE, compressed to pr_id grain ----
_src   = spark.table(SOURCE_TABLE)
_have  = set(_src.columns)
_aggs  = []
def _first(col, alias):
    if col in _have:
        _aggs.append(F.first(col, ignorenulls=True).alias(alias))
_first("Root_Cause_Category",     "gt_root_cause")
_first("Root_Cause_SubCategory",  "gt_root_cause_sub")
_first("Impact_Assessment",       "impact_assessment")
_first("Quality_Final_Assessment", "quality_final")
_first("Program_Number",          "program_number")
_first("Study_Protocol",          "study_protocol")
if "Action_Text" in _have:                                   # row-grain -> distinct joined
    _aggs.append(F.array_join(F.array_distinct(F.collect_list("Action_Text")), "  |  ").alias("action_text"))

src_meta = (_src.groupBy(F.col("Event_Number").cast("string").alias("pr_id")).agg(*_aggs).toPandas())
src_meta["pr_id"] = src_meta["pr_id"].astype(str)
mv = master.merge(src_meta, on="pr_id", how="left")          # mv = master + SME evidence (never mutate master)

# ---- Human vs LLM root-cause alignment (6M taxonomy) ----
LLM_TO_GT = {
    "Man (Human Factors)": "Human Factors", "Method": "Method", "Machine": "Machine",
    "Material": "Material", "Measurement": "Measurement", "Environment": "Environment",
    "Root Cause - Not Identified": "Root Cause - Not Identified",
    "Other / Indeterminate": "Root Cause - Not Identified",
}
for c in ["gt_root_cause", "impact_assessment", "quality_final", "action_text", "program_number", "study_protocol"]:
    if c not in mv:
        mv[c] = np.nan
mv["gt_root_cause"]  = mv["gt_root_cause"].replace("", np.nan)
mv["llm_rc_mapped"]  = mv["root_cause_category"].map(LLM_TO_GT).fillna(mv["root_cause_category"])
mv["rc_agrees"]      = mv["gt_root_cause"].notna() & (mv["llm_rc_mapped"] == mv["gt_root_cause"])
mv["rc_disagrees"]   = mv["gt_root_cause"].notna() & (mv["llm_rc_mapped"] != mv["gt_root_cause"])

# ---- Severity / regulatory-impact flags (from Impact + Quality assessment text) ----
SEVERITY_PATTERNS = {
    "Patient safety":  r"(?i)\b(safety|subject|patient|harm|injur|adverse)",
    "Data integrity":  r"(?i)\b(data integrity|reliabilit|validit|gcp|gxp|falsif|alcoa)",
    "Regulatory":      r"(?i)\b(regulator|authorit|fda|ema|ich|inspection|submission|reportab|recall)",
    "Product quality": r"(?i)\b(product quality|batch|contaminat|out of specification|\boos\b|sterilit)",
}
_imp = (mv["impact_assessment"].fillna("").astype(str) + " " + mv["quality_final"].fillna("").astype(str))
for _name, _pat in SEVERITY_PATTERNS.items():
    mv[f"sev::{_name}"] = _imp.str.contains(_pat, regex=True, na=False)
_sev_cols = [f"sev::{n}" for n in SEVERITY_PATTERNS]
mv["severity_score"] = mv[_sev_cols].sum(axis=1).astype(int)

# ---- Triage / review-queue scoring ----
mv["low_conf"] = mv["confidence"].astype(str).str.lower().isin(["low", "", "nan", "none"]) | mv["confidence"].isna()

def _review_reasons(r):
    out = []
    if r["is_outlier"]:                 out.append("outlier topic")
    if r["low_conf"]:                   out.append("low confidence")
    if r["rc_disagrees"]:               out.append("model\u2260human RC")
    if r["sev::Patient safety"]:        out.append("patient safety")
    if r["sev::Regulatory"]:            out.append("regulatory")
    if r["sev::Data integrity"]:        out.append("data integrity")
    return "; ".join(out)

mv["review_reasons"]  = mv.apply(_review_reasons, axis=1)
mv["review_priority"] = (mv["is_outlier"].astype(int) + mv["low_conf"].astype(int)
                         + mv["rc_disagrees"].astype(int) + mv["severity_score"])

def snippet(text, n=150):
    """Short, single-line event preview from source_free_text_full."""
    t = re.sub(r"\s+", " ", str(text or "")).strip()
    return re.sub(r"^Event_Title:\s*", "", t)[:n]

_valid_rc = mv["gt_root_cause"].notna().sum()
print(f"SME evidence joined for {len(mv):,} events.")
print(f"  Human root cause present : {int(_valid_rc):,}   |   LLM\u2194human agreement: "
      f"{mv.loc[mv['gt_root_cause'].notna(), 'rc_agrees'].mean():.1%}" if _valid_rc else "  Human root cause present : 0")
print(f"  Events with a severity flag: {int((mv['severity_score'] > 0).sum()):,}")
print(f"  Events flagged for review  : {int((mv['review_reasons'] != '').sum()):,}")

### 8.1 · Model vs. human root-cause agreement

Where the LLM's Ishikawa 6M call matches the reviewer-entered `Root_Cause_Category`, and where it diverges. The diagonal is agreement; off-diagonal cells are the events worth auditing. Both are mapped to the same 6M naming before comparison.

In [0]:
# Cell 18 — Figure H: model vs. human root-cause agreement
valid = mv[mv["gt_root_cause"].notna()].copy()

if len(valid):
    agree = valid["rc_agrees"].mean()
    cm = pd.crosstab(valid["gt_root_cause"], valid["llm_rc_mapped"])
    order = sorted(set(cm.index) | set(cm.columns))
    cm = cm.reindex(index=order, columns=order, fill_value=0)

    # per-category recall (diagonal / row total)
    diag = pd.Series({c: cm.loc[c, c] if c in cm.columns else 0 for c in cm.index})
    per_cat = (diag / cm.sum(axis=1).replace(0, np.nan)).fillna(0).sort_values()

    fig, axes = plt.subplots(1, 2, figsize=(19, 7), gridspec_kw={"width_ratios": [1.35, 1]})
    fig.suptitle(f"Model vs. Human Root Cause  —  overall agreement {agree:.1%}  "
                 f"(n={len(valid):,} with human label)", fontsize=15, fontweight="bold")

    # confusion matrix
    ax = axes[0]
    im = ax.imshow(cm.values, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=35, ha="right", fontsize=9)
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=9)
    ax.set_xlabel("LLM (mapped to 6M)"); ax.set_ylabel("Human Root_Cause_Category")
    ax.set_title("Confusion matrix"); ax.grid(False)
    vmax = cm.values.max()
    for i in range(len(order)):
        for j in range(len(order)):
            v = int(cm.values[i, j])
            if v:
                ax.text(j, i, v, ha="center", va="center", fontsize=8,
                        color="white" if v > vmax * 0.6 else "#333")
    fig.colorbar(im, ax=ax, shrink=0.8, label="Events")

    # per-category agreement
    axes[1].barh(range(len(per_cat)), per_cat.values,
                 color=["#C44E52" if v < 0.5 else "#55A868" for v in per_cat.values])
    axes[1].set_yticks(range(len(per_cat))); axes[1].set_yticklabels(per_cat.index, fontsize=9)
    axes[1].set_xlim(0, 1); axes[1].set_title("Per-category agreement (human recall)")
    axes[1].set_xlabel("Share matched by LLM")
    for i, v in enumerate(per_cat.values):
        axes[1].text(v, i, f"  {v:.0%}", va="center", fontsize=8)

    plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show()
    print(f"Agreement {agree:.1%} on {len(valid):,} events. "
          f"{int(valid['rc_disagrees'].sum()):,} disagreements are surfaced in the review queue (8.3).")
else:
    print("No human Root_Cause_Category available to compare against.")

### 8.2 · Severity & regulatory-impact flags

Keyword flags over `Impact_Assessment` + `Quality_Final_Assessment` surface the events that carry patient-safety, data-integrity, regulatory, or product-quality weight — so reviewers triage by risk, not just volume. The table below is the high-severity worklist (sortable/filterable).

In [0]:
# Cell 19 — Figure I: severity flags + high-severity worklist
sev_counts = pd.Series({n: int(mv[f"sev::{n}"].sum()) for n in SEVERITY_PATTERNS}).sort_values()
any_flag   = int((mv["severity_score"] > 0).sum())

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Severity & Regulatory-Impact Flags", fontsize=15, fontweight="bold")

# flag counts
axes[0].barh(range(len(sev_counts)), sev_counts.values, color=["#C44E52", "#8172B3", "#DD8452", "#4C72B0"][:len(sev_counts)])
axes[0].set_yticks(range(len(sev_counts))); axes[0].set_yticklabels(sev_counts.index)
axes[0].set_title("Events per severity flag"); axes[0].set_xlabel("Events")
for i, v in enumerate(sev_counts.values):
    axes[0].text(v, i, f"  {v:,}", va="center", fontsize=8)
axes[0].margins(x=0.14)

# any-flag donut
axes[1].pie([any_flag, N - any_flag], labels=["Has severity flag", "No flag"],
            autopct=lambda p: f"{p:.0f}%\n({int(round(p * N / 100)):,})",
            colors=["#C44E52", "#cccccc"], startangle=90,
            wedgeprops=dict(width=0.42, edgecolor="white"), textprops=dict(fontsize=9))
axes[1].set_title(f"Corpus severity coverage (n={N:,})")

# severity-score distribution
score_dist = mv["severity_score"].value_counts().sort_index()
axes[2].bar(score_dist.index, score_dist.values, color="#8172B3", edgecolor="white")
axes[2].set_title("# severity flags per event"); axes[2].set_xlabel("Flag count"); axes[2].set_ylabel("Events")
axes[2].set_xticks(range(0, len(SEVERITY_PATTERNS) + 1))
for x, v in zip(score_dist.index, score_dist.values):
    axes[2].text(x, v, f"{v:,}", ha="center", va="bottom", fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show()

# ---- High-severity worklist (the events behind the flags) ----
hi_sev = (mv[mv["severity_score"] > 0]
          .assign(event=lambda d: d["source_free_text_full"].apply(snippet),
                  flags=lambda d: d.apply(
                      lambda r: ", ".join(n for n in SEVERITY_PATTERNS if r[f"sev::{n}"]), axis=1))
          .sort_values(["severity_score", "review_priority"], ascending=False)
          [["pr_id", "severity_score", "flags", "topic_label", "root_cause_category",
            "gt_root_cause", "confidence", "event"]]
          .head(300))
print(f"High-severity worklist: {len(mv[mv['severity_score'] > 0]):,} events (showing top {len(hi_sev):,}).")
display(hi_sev)

### 8.3 · Review & validation queue

A concrete worklist instead of "route N to review." Each event is scored by why it needs a human: outlier topic, low model confidence, model≠human root cause, or a severity flag. Highest-priority first.

In [0]:
# Cell 20 — Figure J: review-queue drivers + prioritized worklist
reason_counts = (mv.loc[mv["review_reasons"] != "", "review_reasons"]
                 .str.split("; ").explode().value_counts())
queue = (mv[mv["review_reasons"] != ""]
         .assign(event=lambda d: d["source_free_text_full"].apply(snippet))
         .sort_values("review_priority", ascending=False)
         [["pr_id", "review_priority", "review_reasons", "topic_label",
           "root_cause_category", "gt_root_cause", "confidence", "severity_score", "event"]])

fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle("Why events land in the review queue", fontsize=14, fontweight="bold")
rc = reason_counts[::-1]
ax.barh(range(len(rc)), rc.values, color="#DD8452")
ax.set_yticks(range(len(rc))); ax.set_yticklabels(rc.index)
ax.set_xlabel("Events"); ax.grid(True, axis="x")
for i, v in enumerate(rc.values):
    ax.text(v, i, f"  {v:,}", va="center", fontsize=8)
ax.margins(x=0.14)
plt.tight_layout(rect=[0, 0, 1, 0.92]); plt.show()

print(f"Review queue: {len(queue):,} events flagged "
      f"({100 * len(queue) / N:.0f}% of corpus). Auto-accept the remaining {N - len(queue):,}.")
display(queue.head(400))

### 8.4 · Drill-down worklists — the events behind the bars

Every headline bar expands to its underlying records: the deviations inside each recurring *topic ▸ root-cause* signature, the deviations citing each most-deviated SOP, and the deviations inside each repeat-offender program. These `display()` tables are sortable and filterable in Databricks.

In [0]:
# Cell 21 — Drill-down worklists: events behind the signature / SOP / program bars
mv["event"]     = mv["source_free_text_full"].apply(snippet)
mv["signature"] = mv["topic_label"].str.slice(0, 28) + "  \u25b8  " + mv["root_cause_category"].fillna("\u2014")

# (a) recurring topic > root-cause signatures -> member events
_ok = ~mv["is_outlier"] & mv["root_cause_category"].notna()
top_sigs = mv.loc[_ok, "signature"].value_counts().head(8)
sig_members = (mv[mv["signature"].isin(top_sigs.index) & _ok]
               [["signature", "pr_id", "confidence", "gt_root_cause", "event"]]
               .sort_values(["signature", "confidence"]))
print(f"(a) Top {len(top_sigs)} recurring signatures \u2192 {len(sig_members):,} member events:")
display(sig_members)

# (b) most-deviated SOPs -> the deviations citing each
_sop_rows = [(str(d), r.pr_id) for r in mv.itertuples(index=False)
             for d in to_list(getattr(r, "documents", None)) if str(d).upper().startswith("SOP")]
sop_map = pd.DataFrame(_sop_rows, columns=["sop", "pr_id"])
if len(sop_map):
    top_sops = sop_map["sop"].value_counts().head(8)
    sop_members = (sop_map[sop_map["sop"].isin(top_sops.index)]
                   .merge(mv[["pr_id", "topic_label", "root_cause_category", "event"]], on="pr_id", how="left")
                   .sort_values("sop"))
    print(f"(b) Top {len(top_sops)} most-deviated SOPs \u2192 {len(sop_members):,} citing events:")
    display(sop_members)
else:
    print("(b) No SOP references found.")

# (c) repeat-offender programs -> member events
prog = mv["program_number"].dropna().astype(str).str.strip()
prog = prog[~prog.str.upper().isin({"", "N/A", "NA", "NONE", "NULL", "-", "--"})]
recur = prog.value_counts()
recur = recur[recur >= 2].head(10)
if len(recur):
    prog_members = (mv[mv["program_number"].astype(str).str.strip().isin(recur.index)]
                    [["program_number", "pr_id", "topic_label", "root_cause_category", "gt_root_cause", "event"]]
                    .sort_values("program_number"))
    print(f"(c) {len(recur)} repeat-offender programs (\u22652 deviations) \u2192 {len(prog_members):,} member events:")
    display(prog_members)
else:
    print("(c) No programs with \u22652 deviations.")

## 9 · Interactive filters & curated triage export

Self-serve for SMEs: a Databricks widget bar to slice the corpus by topic category, root cause, severity, review priority, or a free-text keyword — and a curated per-event `deviation_triage` Delta table (one tidy row per deviation) they can query, pivot, or export to CSV.

In [0]:
# Cell 22 — Create the filter widget bar (renders at the top of the notebook)
dbutils.widgets.removeAll()

_cat_choices = ["(All)"] + sorted(x for x in mv["topic_category"].dropna().unique())
_rc_choices  = ["(All)"] + sorted(x for x in mv["root_cause_category"].dropna().unique())[:40]
_sev_choices = ["(All)", "Has any flag"] + list(SEVERITY_PATTERNS)

dbutils.widgets.dropdown("flt_category",    "(All)",      _cat_choices,                        "Topic category")
dbutils.widgets.dropdown("flt_rootcause",   "(All)",      _rc_choices,                         "Root cause (LLM)")
dbutils.widgets.dropdown("flt_severity",    "(All)",      _sev_choices,                        "Severity flag")
dbutils.widgets.dropdown("flt_minpriority", "0",          ["0", "1", "2", "3", "4", "5"],      "Min review priority")
dbutils.widgets.dropdown("flt_reviewonly",  "All events", ["All events", "Review queue only"], "Scope")
dbutils.widgets.text("flt_keyword", "", "Keyword in event text")

print("Filter widgets ready at the top of the notebook. Adjust them, then re-run the next cell.")
print("Remove them anytime with:  dbutils.widgets.removeAll()")

In [0]:
# Cell 23 — Apply the widget filters and show the filtered worklist
_w = {k: dbutils.widgets.get(k) for k in
      ["flt_category", "flt_rootcause", "flt_severity", "flt_minpriority", "flt_reviewonly", "flt_keyword"]}

f = mv
if _w["flt_category"]  != "(All)":       f = f[f["topic_category"] == _w["flt_category"]]
if _w["flt_rootcause"] != "(All)":       f = f[f["root_cause_category"] == _w["flt_rootcause"]]
if _w["flt_severity"]  == "Has any flag": f = f[f["severity_score"] > 0]
elif _w["flt_severity"] in SEVERITY_PATTERNS: f = f[f[f"sev::{_w['flt_severity']}"]]
f = f[f["review_priority"] >= int(_w["flt_minpriority"])]
if _w["flt_reviewonly"] == "Review queue only": f = f[f["review_reasons"] != ""]
_kw = _w["flt_keyword"].strip()
if _kw:
    f = f[f["source_free_text_full"].str.contains(re.escape(_kw), case=False, na=False)]

view = (f.assign(event=lambda d: d["source_free_text_full"].apply(snippet))
         .sort_values("review_priority", ascending=False)
         [["pr_id", "review_priority", "review_reasons", "topic_label", "topic_category",
           "root_cause_category", "gt_root_cause", "confidence", "severity_score", "event"]])

print(f"Filter \u2192 {len(view):,} / {len(mv):,} events   "
      f"[category={_w['flt_category']} | rootcause={_w['flt_rootcause']} | severity={_w['flt_severity']} | "
      f"min_priority={_w['flt_minpriority']} | scope={_w['flt_reviewonly']} | keyword='{_kw}']")
display(view.head(500))

In [0]:
# Cell 24 — Curated per-event triage table (Delta) for SME self-serve + CSV export
from pyspark.sql import functions as F

TRIAGE_TABLE = f"{CATALOG}.{MART}.deviation_triage"

triage = pd.DataFrame({
    "pr_id":            mv["pr_id"],
    "topic_label":      mv["topic_label"],
    "topic_category":   mv["topic_category"],
    "llm_root_cause":   mv["root_cause_category"],
    "human_root_cause": mv["gt_root_cause"],
    "rc_agrees":        mv["rc_agrees"],
    "confidence":       mv["confidence"],
    "severity_score":   mv["severity_score"],
    "severity_flags":   mv.apply(lambda r: ", ".join(n for n in SEVERITY_PATTERNS if r[f"sev::{n}"]), axis=1),
    "review_priority":  mv["review_priority"],
    "review_reasons":   mv["review_reasons"],
    "clinical_ids":     mv["clinical_ids"].apply(lambda x: ", ".join(to_list(x))),
    "documents":        mv["documents"].apply(lambda x: ", ".join(to_list(x))),
    "program_number":   mv["program_number"],
    "study_protocol":   mv["study_protocol"],
    "capa_action_text": mv["action_text"],
    "event_preview":    mv["source_free_text_full"].apply(snippet),
})

_str_cols = ["pr_id", "topic_label", "topic_category", "llm_root_cause", "human_root_cause", "confidence",
             "severity_flags", "review_reasons", "clinical_ids", "documents", "program_number",
             "study_protocol", "capa_action_text", "event_preview"]
triage[_str_cols] = triage[_str_cols].fillna("")

# write to MART + grant SELECT to the read group
write_and_grant(spark.createDataFrame(triage), "deviation_triage")

# keep the descriptive comment (helper doesn't set comments)
spark.sql(f"COMMENT ON TABLE {TRIAGE_TABLE} IS "
          "'Per-event SME triage: topic + category, LLM & human root cause + agreement, severity flags, "
          "review priority/reasons, resolved entities, program/protocol, CAPA text. Built by deviation_statistics.'")

print(f"Wrote {TRIAGE_TABLE}: {spark.table(TRIAGE_TABLE).count():,} rows x {len(triage.columns)} cols "
      "(use the download button on the table below to export CSV).")
display(spark.table(TRIAGE_TABLE).orderBy(F.col("review_priority").desc()).limit(300))

### Appendix · optional live hybrid semantic search

Optional demonstration that queries the **already-built** AI Search fine index live (reusing the BGE-M3 encoder + vector endpoint) for ad-hoc protocol-deviation and unblinding queries. Off by default because it loads a ~3 GB model and needs the endpoint online. Set `RUN_LIVE_SEARCH = True` to run.

In [0]:
# Cell 17 — (Optional) live hybrid semantic search over the EXISTING AI Search index
# Reuses the already-built fine index + BGE-M3 query encoder. No raw reprocessing.
RUN_LIVE_SEARCH = False
QUERIES = {
    "protocol deviation": "clinical protocol deviation or non-compliance with study procedures",
    "unblinding":         "site inadvertently disclosed treatment assignment, possible unblinding",
}

if RUN_LIVE_SEARCH:
    import os, sys
    from FlagEmbedding import BGEM3FlagModel
    import databricks
    for _sp in sys.path:
        _dbp = os.path.join(_sp, "databricks")
        if os.path.isdir(os.path.join(_dbp, "ai_search")) and _dbp not in databricks.__path__:
            databricks.__path__.append(_dbp); break
    from databricks.ai_search.client import AISearchClient

    _bge    = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
    _client = AISearchClient()
    _index  = _client.get_index(endpoint_name=VS_ENDPOINT, index_name=EMB_INDEX_FINE)
    _text   = {r["pr_id"]: (r["source_free_text_full"] or "")
               for r in spark.table(EMBED_INPUT).select("pr_id", "source_free_text_full").collect()}

    for name, q in QUERIES.items():
        qv = _bge.encode([q], max_length=MAX_LEN, return_dense=True,
                         return_sparse=False, return_colbert_vecs=False)["dense_vecs"][0].tolist()
        res = _index.similarity_search(query_vector=qv, query_text=q, query_type="HYBRID",
                                       columns=["pr_id"], num_results=10)
        rows = res.get("result", {}).get("data_array", []) if isinstance(res, dict) else []
        print(f"\n=== {name.upper()} — top {len(rows)} hits ===")
        for i, row in enumerate(rows, 1):
            pid = str(row[0]); snip = _text.get(pid, "").replace("\n", " ")[:110]
            print(f"  {i:2d}. [{pid}] {snip}")
else:
    print("RUN_LIVE_SEARCH is False — cohorts above come from pre-written tables only. "
          "Set it to True to query the live vector index.")